# Phase B+C — PSQI 4 차원 학습·평가 노트북

검증 게이트 #1을 *셀 단위 인터랙티브*로 확인하기 위한 노트북.

**합격 기준**:
- C1·C2·C3·C4 *각각이* 차원별 naive baseline 이김
- 합산 RMSE < 합산 naive baseline
- 4 차원 SHAP이 *서로 다른 변수* top-1으로 잡음 (Phase C)

**관련 모듈**:
- `experiments/lib/datasets.py` — master 데이터 빌드 (24 features + 5 PSQI targets)
- `experiments/lib/models.py` — 학습·평가 함수
- `experiments/lib/psqi.py` — 4 차원 점수화
- `experiments/lib/features.py` — 저녁 세분화

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from experiments.lib import datasets as D
from experiments.lib import models as M

DATA_DIR = ROOT / 'JG-Data'
FEEDBACK_CSV = ROOT / 'db' / 'init' / 'feedback_sleep.csv'
print(f'ROOT     = {ROOT}')
print(f'DATA_DIR = {DATA_DIR}')

## 1. 데이터 로딩 + master 빌드

In [ ]:
df = D.build_master_dataset(DATA_DIR, feedback_csv=FEEDBACK_CSV)
print(f'rows = {len(df)}, columns = {len(df.columns)}')
df.head()

## 2. 4 차원 점수 분포 + 결측률

In [ ]:
score_cols = ['c1_subjective', 'c2_latency', 'c3_duration', 'c4_efficiency', 'score_total']
summary = df[score_cols].describe().T
summary['missing_pct'] = (df[score_cols].isna().sum() / len(df) * 100).values
summary

In [ ]:
# 차원별 점수 분포 시각화
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, col in zip(axes, ['c1_subjective', 'c2_latency', 'c3_duration', 'c4_efficiency']):
    df[col].dropna().value_counts().sort_index().plot(kind='bar', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('score (0~3)')
plt.tight_layout()
plt.show()

## 3. Phase B — 4 차원 학습·평가 + naive baseline 비교

시간 holdout: 마지막 7일을 test, 그 이전을 train.
차원별로 별도 CatBoost 모델, target NaN 행은 자동 제외.

In [ ]:
feature_cols = [c for c in D.FEATURE_COLUMNS if c in df.columns]
print(f'features 사용: {len(feature_cols)}개')
for c in feature_cols:
    print(f'  - {c}')

In [ ]:
results, dim_models = M.train_eval_psqi_4dim(
    df, feature_cols=feature_cols, dims=M.PSQI_DIMS, holdout_days=7,
)
print(M.summarize_results(results))

## 4. Phase C — 차원별 SHAP top-3

*4 차원이 서로 다른 변수를 top-1으로 잡는가*가 분리 모델의 *해석 우위* 입증의 핵심.

In [ ]:
import shap

shap_top_by_dim = {}
for dim, model in dim_models.items():
    valid = df.dropna(subset=[dim])
    X = valid[feature_cols]
    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X)
    mean_abs = np.abs(shap_vals).mean(axis=0)
    top_idx = np.argsort(mean_abs)[::-1][:3]
    top_feats = [(feature_cols[i], float(mean_abs[i])) for i in top_idx]
    shap_top_by_dim[dim] = top_feats
    print(f'{dim}: ' + ', '.join(f'{f}({v:.3f})' for f, v in top_feats))

In [ ]:
# 4 차원 top-1 변수가 *서로 다른가*
top1 = {dim: feats[0][0] for dim, feats in shap_top_by_dim.items()}
n_unique = len(set(top1.values()))
print(f'4 차원 top-1 변수: {top1}')
print(f'고유 변수 수: {n_unique} / 4')
print(f'분리 모델의 *해석 우위* {"✅ 입증" if n_unique >= 3 else "❌ 약화"}')

## 5. 게이트 #1 종합 판정

In [ ]:
n_pass_dims = sum(1 for ev in results.by_dim.values() if ev.beats_baseline)
total_pass = (
    results.total_model_rmse is not None
    and results.total_baseline_rmse is not None
    and results.total_model_rmse < results.total_baseline_rmse
)
shap_pass = n_unique >= 3

print('━' * 60)
print('검증 게이트 #1 — 종합 판정')
print('━' * 60)
print(f'  차원별 baseline 이김: {n_pass_dims}/{len(results.by_dim)}  → {"✅" if n_pass_dims == len(results.by_dim) else "❌"}')
print(f'  합산 baseline 이김:   {"✅" if total_pass else "❌"}')
print(f'  4 차원 SHAP 분리:     {"✅" if shap_pass else "❌"}')
print('━' * 60)
verdict = (n_pass_dims == len(results.by_dim)) and total_pass and shap_pass
print(f'  최종: {"✅ PASS — Phase D 운영 통합 진행 가능" if verdict else "❌ FAIL — 후속 액션 필요"}')